# CEG-WM Content V6 — detector-domain ISS gain/target fit

Initial user-only GPU handoff for the frozen 32-unit development fit. Set the Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN`, then run once from top to bottom. Results remain as one create-only JSON/SHA pair in Google Drive. Stop after any failure; there is no retry or resume.

## 1. Mount Drive and check out the frozen exact

In [ ]:
from google.colab import drive

HANDOFF_FAILED = False
try:
    drive.mount("/content/drive")
except BaseException:
    HANDOFF_FAILED = True
    print('CEGWM_CONTENT_V6_ISS_FIT_HANDOFF_FAILURE {"producer_exact":"70d4147ceb9832acf7511b2e68edf0c47e453229","stage":"drive_mount","status":"operational_failure"}', flush=True)

import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-v6-detector-domain-iss"
EXACT = "70d4147ceb9832acf7511b2e68edf0c47e453229"
RUNNER_MODULE = "experiments.run_content_v6_iss_fit"
ASSET_FILENAME = "content_v6_iss_gain_target_v1.json"
RECEIPT_PREFIX = "CEGWM_CONTENT_V6_ISS_FIT_RECEIPT"
FAILURE_PREFIX = "CEGWM_CONTENT_V6_ISS_FIT_HANDOFF_FAILURE"
SPEC_SHA256 = "770a0d79cfdb9d98156f6b8d585ae0c0554313f5dfd745ceb5e228d7f3fc02ce"

repo = pathlib.Path("/content/cegwm-stage-a-content-v6-iss-fit-source")
local_root = pathlib.Path("/content/cegwm-stage-a-content-v6-iss-fit-local")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v6_iss_fit")
bound_result_dir = artifact_sink / EXACT
asset_path = bound_result_dir / ASSET_FILENAME
checksum_path = bound_result_dir / (ASSET_FILENAME + ".sha256")

RUNNER_ATTEMPTED = False

def fail(stage):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    HANDOFF_FAILED = True
    payload = {"status": "operational_failure", "producer_exact": EXACT, "stage": stage}
    print(FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True, capture_output=True, text=True
    ).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if repo.exists():
            raise FileExistsError
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain")
        ):
            raise RuntimeError
    except BaseException:
        fail("source_checkout")

## 2. Install the checked-out project

In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain")
        ):
            raise RuntimeError
    except BaseException:
        fail("dependency_install")

## 3. Run the fit producer exactly once

This cell passes both Secrets only to the child process, drains stdout into a fixed cap, suppresses raw stderr, and prints either one validated receipt or one sanitized failure line.

In [ ]:
import os
import re
from google.colab import userdata

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ""
    hf_token = ""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_failed = False
    CAPTURE_LIMIT = 4096
    try:
        if local_root.exists() or bound_result_dir.exists() or asset_path.exists() or checksum_path.exists():
            raise FileExistsError
        local_root.mkdir(parents=True, exist_ok=False)
        hf_cache = local_root / "hf-cache"
        hf_cache.mkdir(exist_ok=False)
        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key:
            raise RuntimeError
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain")
            or bound_result_dir.exists()
        ):
            raise RuntimeError
        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        runner_env["HF_HOME"] = str(hf_cache)
        root_key = ""
        hf_token = ""
        process = subprocess.Popen(
            [
                sys.executable, "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
                "--artifact-sink", str(artifact_sink),
            ],
            cwd=repo, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        )
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
        runner_rc = process.wait()
    except BaseException:
        launch_failed = True
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ""
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
            runner_env.pop("HF_HOME", None)
        runner_env = None

    if launch_failed:
        captured.clear()
        fail("fit_runner_launch")
    elif runner_rc != 0:
        captured.clear()
        fail("fit_runner_nonzero")
    elif capture_overflow:
        captured.clear()
        fail("fit_runner_stdout_overflow")
    else:
        try:
            captured_text = captured.decode("utf-8", errors="strict")
            if captured_text.count("\n") != 1 or not captured_text.endswith("\n"):
                raise RuntimeError
            line = captured_text[:-1]
            if not line.startswith(RECEIPT_PREFIX + " "):
                raise RuntimeError
            receipt = json.loads(line.split(" ", 1)[1])
            if (
                not isinstance(receipt, dict)
                or set(receipt) != {"asset_sha256", "development_public_key_digest", "fit_sample_count", "personal_spec_sha256", "producer_exact"}
                or re.fullmatch(r"[0-9a-f]{64}", receipt.get("asset_sha256", "")) is None
                or re.fullmatch(r"[0-9a-f]{64}", receipt.get("development_public_key_digest", "")) is None
                or receipt.get("fit_sample_count") != 32
                or receipt.get("personal_spec_sha256") != SPEC_SHA256
                or receipt.get("producer_exact") != EXACT
            ):
                raise RuntimeError
            print(line, flush=True)
        except BaseException:
            fail("fit_runner_receipt")
        finally:
            captured.clear()

## 4. Authenticate the existing Drive pair

This runner-free cell only reads the exact-bound pair, verifies the sidecar against the JSON bytes, and prints one bounded Drive artifact receipt. It never downloads, rewrites, resumes, or reruns anything.

In [ ]:
import hashlib
import json
import pathlib
import re
from google.colab import drive

EXACT = "70d4147ceb9832acf7511b2e68edf0c47e453229"
ASSET_FILENAME = "content_v6_iss_gain_target_v1.json"
FAILURE_PREFIX = "CEGWM_CONTENT_V6_ISS_FIT_HANDOFF_FAILURE"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V6_ISS_FIT_ARTIFACT"
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v6_iss_fit")
result_dir = artifact_sink / EXACT
asset_path = result_dir / ASSET_FILENAME
checksum_path = result_dir / (ASSET_FILENAME + ".sha256")

if not bool(globals().get("HANDOFF_FAILED", False)):
    artifact_failed = False

    def artifact_fail(stage):
        global artifact_failed
        if artifact_failed:
            return
        artifact_failed = True
        payload = {"status": "operational_failure", "producer_exact": EXACT, "stage": stage}
        print(FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

    try:
        if not pathlib.Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        expected_names = sorted([ASSET_FILENAME, ASSET_FILENAME + ".sha256"])
        if (
            not result_dir.is_dir()
            or not asset_path.is_file()
            or not checksum_path.is_file()
            or sorted(path.name for path in result_dir.iterdir()) != expected_names
        ):
            raise RuntimeError
        asset_bytes = asset_path.read_bytes()
        sidecar = checksum_path.read_text(encoding="ascii")
        if re.fullmatch(r"[0-9a-f]{64}  " + re.escape(ASSET_FILENAME) + r"\n", sidecar) is None:
            raise RuntimeError
        asset_sha256 = hashlib.sha256(asset_bytes).hexdigest()
        if asset_sha256 != sidecar[:64]:
            raise RuntimeError
        asset_bytes = b""
        payload = {
            "asset_path": str(asset_path),
            "asset_sha256": asset_sha256,
            "producer_exact": EXACT,
            "sidecar_path": str(checksum_path),
            "status": "artifact_pair_saved",
        }
        print(ARTIFACT_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)
    except BaseException:
        artifact_fail("existing_asset_pair_validation")